# 02 · Datasets and Tables

> **Module 7 · Python Client Library · Lectures 7.2 and 7.3**

In this notebook you will:

- Create datasets with labels, descriptions, and expirations
- Build schemas with `SchemaField`, including nested and repeated fields
- Create partitioned and clustered tables
- Load data and make the whole flow idempotent

---

## Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, dataset_exists

from google.cloud import bigquery
from google.cloud.exceptions import NotFound

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
client = get_client(project=PROJECT_ID)
print(f"Project: {client.project}")

## 1. Datasets — the option map

| Python attribute | SQL equivalent |
|------------------|----------------|
| `dataset_id` | The dataset name |
| `location` | `OPTIONS(location = ...)` |
| `description` | `OPTIONS(description = ...)` |
| `labels` | `OPTIONS(labels = ...)` |
| `default_table_expiration_ms` | `OPTIONS(default_table_expiration_days = ...)` |
| `default_partition_expiration_ms` | `OPTIONS(default_partition_expiration_days = ...)` |
| `is_case_insensitive` | `OPTIONS(is_case_insensitive = ...)` |
| `default_collation` | `OPTIONS(default_collation = ...)` |
| `access_entries` | IAM bindings |

### 💡 The idempotency key

`client.create_dataset(dataset, exists_ok=True)` succeeds whether or not the dataset
exists. **Always use it** in setup scripts.

In [ ]:
DATASETS = {
    "raw": {
        "description": "Immutable raw landing zone. Never update, never delete manually.",
        "default_partition_expiration_days": 180,
        "labels": {"tier": "raw", "env": "prod", "owner": "data-eng"},
    },
    "curated": {
        "description": "Curated, conformed analytics models",
        "default_table_expiration_days": 365,
        "default_partition_expiration_days": 730,
        "labels": {"tier": "curated", "env": "prod", "owner": "data-eng"},
    },
    "mart": {
        "description": "BI serving layer",
        "labels": {"tier": "gold", "env": "prod", "owner": "analytics-eng"},
    },
    "scratch": {
        "description": "Experiments. Auto-expires in 7 days.",
        "default_table_expiration_days": 7,
        "labels": {"tier": "scratch", "env": "dev"},
    },
}

MS_PER_DAY = 24 * 60 * 60 * 1000

for dataset_id, config in DATASETS.items():
    kwargs = {
        "location": "US",
        "description": config["description"],
        "labels": config.get("labels", {}),
    }
    if "default_table_expiration_days" in config:
        kwargs["default_table_expiration_ms"] = config["default_table_expiration_days"] * MS_PER_DAY
    if "default_partition_expiration_days" in config:
        kwargs["default_partition_expiration_ms"] = config["default_partition_expiration_days"] * MS_PER_DAY

    dataset = bigquery.Dataset(f"{PROJECT_ID}.{dataset_id}", **kwargs)
    dataset = client.create_dataset(dataset, exists_ok=True)
    print(f"✓ {dataset.full_dataset_id}")

In [ ]:
# List the datasets with their metadata
rows = []
for dataset in client.list_datasets(project=PROJECT_ID):
    labels = dataset.labels or {}
    rows.append(
        {
            "dataset_id": dataset.dataset_id,
            "location": dataset.location,
            "owner": labels.get("owner", "-"),
            "tier": labels.get("tier", "-"),
            "table_exp_days": (
                round(dataset.default_table_expiration_ms / MS_PER_DAY)
                if dataset.default_table_expiration_ms else None
            ),
        }
    )

display(run_to_dataframe("SELECT * FROM UNNEST(@rows)", params={"rows": rows}))

In [ ]:
# Existence check and update
print(f"curated exists: {dataset_exists(f'{PROJECT_ID}.curated')}")
print(f"nope exists:    {dataset_exists(f'{PROJECT_ID}.nope')}")

dataset = client.get_dataset(f"{PROJECT_ID}.curated")
dataset.description = "Curated models (updated)"
dataset.labels = {**(dataset.labels or {}), "reviewed": "2026-10"}
dataset = client.update_dataset(dataset, update_mode="update_fields")
print(f"\nUpdated: {dataset.description}")
print(f"Labels:  {dataset.labels}")

## 2. Schemas with `SchemaField`

`SchemaField(name, field_type, mode, description, fields)` — the `fields` argument is
how you build a nested `RECORD`.

In [ ]:
ORDER_SCHEMA = [
    bigquery.SchemaField("order_id", "STRING", mode="REQUIRED", description="Unique order ID"),
    bigquery.SchemaField("customer_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("order_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField("order_ts", "TIMESTAMP"),
    bigquery.SchemaField("total_amount", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("is_completed", "BOOLEAN"),
    bigquery.SchemaField(
        name="line_items",
        field_type="RECORD",
        mode="REPEATED",
        fields=[
            bigquery.SchemaField("product_id", "STRING"),
            bigquery.SchemaField("quantity", "INTEGER"),
            bigquery.SchemaField("unit_price", "NUMERIC"),
        ],
    ),
]

for field in ORDER_SCHEMA:
    nested = f"  -> {[f.name for f in (field.fields or [])]}" if field.fields else ""
    print(f"{field.name:14} {field.field_type:10} {field.mode:10}{nested}")

In [ ]:
# Loading a schema from a JSON file — the recommended pattern
import json

SCHEMA_JSON = [
    {"name": "order_id",    "type": "STRING",    "mode": "REQUIRED", "description": "Unique order ID"},
    {"name": "customer_id", "type": "STRING",    "mode": "REQUIRED"},
    {"name": "order_date",  "type": "DATE",      "mode": "REQUIRED"},
    {"name": "total_amount","type": "NUMERIC",   "mode": "REQUIRED"},
    {"name": "status",      "type": "STRING",    "mode": "REQUIRED"},
    {"name": "line_items",  "type": "RECORD",    "mode": "REPEATED",
     "fields": [
        {"name": "product_id", "type": "STRING"},
        {"name": "quantity",   "type": "INTEGER"},
        {"name": "unit_price", "type": "NUMERIC"},
     ]},
]


def schema_from_json(raw: list[dict]) -> list[bigquery.SchemaField]:
    """Convert a JSON schema definition into SchemaField objects."""
    return [
        bigquery.SchemaField(
            name=f["name"],
            field_type=f["type"],
            mode=f.get("mode", "NULLABLE"),
            description=f.get("description"),
            fields=[
                bigquery.SchemaField(
                    name=s["name"],
                    field_type=s["type"],
                    mode=s.get("mode", "NULLABLE"),
                )
                for s in f.get("fields", [])
            ] or None,
        )
        for f in raw
    ]


with open("schemas_orders.json", "w") as f:
    json.dump(SCHEMA_JSON, f, indent=2)

loaded = schema_from_json(SCHEMA_JSON)
print("✅ Schema round-trip:")
for field in loaded:
    nested = f"  -> {[f.name for f in (field.fields or [])]}" if field.fields else ""
    print(f"{field.name:14} {field.field_type:10} {field.mode:10}{nested}")

## 3. Create a partitioned, clustered table

```python
table = bigquery.Table(f"{PROJECT_ID}.curated.orders", schema=ORDER_SCHEMA)
table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="order_date",
    require_partition_filter=True,
)
table.clustering_fields = ["customer_id"]
client.create_table(table, exists_ok=True)
```

In [ ]:
table = bigquery.Table(f"{PROJECT_ID}.curated.orders", schema=ORDER_SCHEMA)
table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="order_date",
    expiration_ms=730 * MS_PER_DAY,
    require_partition_filter=True,
)
table.clustering_fields = ["customer_id", "status"]
table.description = "Conformed orders"
table.labels = {"tier": "silver", "owner": "data-eng"}

table = client.create_table(table, exists_ok=True)
print(f"✓ {table.full_table_id}")
print(f"  partition: {table.time_partitioning.type_} on {table.time_partitioning.field}")
print(f"  clustering: {table.clustering_fields}")
print(f"  require filter: {table.time_partitioning.require_partition_filter}")

In [ ]:
# The SQL equivalent, for reference
SQL_EQUIVALENT = """
CREATE OR REPLACE TABLE `YOUR_PROJECT_ID.curated.orders` (
  order_id     STRING NOT NULL,
  customer_id  STRING NOT NULL,
  order_date   DATE   NOT NULL,
  order_ts     TIMESTAMP,
  total_amount NUMERIC NOT NULL,
  status       STRING NOT NULL,
  is_completed BOOLEAN,
  line_items   ARRAY<STRUCT<product_id STRING, quantity INT64, unit_price NUMERIC>>
)
PARTITION BY order_date
CLUSTER BY customer_id, status
OPTIONS(
  description = "Conformed orders",
  require_partition_filter = TRUE,
  partition_expiration_days = 730,
  labels = [("tier", "silver"), ("owner", "data-eng")]
)
"""

print(SQL_EQUIVALENT)

## 4. Load data

Three load paths:

- `load_table_from_file` — a local file
- `load_table_from_dataframe` — a pandas DataFrame
- `load_table_from_uri` — Cloud Storage

> ⚠️ **Always call `job.result()`.** The load is asynchronous. Without it your script
> exits and the load may be cancelled. This is the number one cause of "the load did
> not work".

In [ ]:
import pandas as pd

df = pd.DataFrame(
    {
        "order_id":     [f"o{i:04d}" for i in range(1, 51)],
        "customer_id":  [f"c{i % 7:04d}" for i in range(1, 51)],
        "order_date":   ["2026-01-15"] * 50,
        "total_amount": [round(10 + i * 3.5, 2) for i in range(1, 51)],
        "status":       ["active" if i % 3 else "complete" for i in range(1, 51)],
    }
)
df["order_date"] = pd.to_datetime(df["order_date"]).dt.date

job_config = bigquery.LoadJobConfig(
    schema=ORDER_SCHEMA,
    source_format=bigquery.SourceFormat.CSV,
    write_disposition="WRITE_TRUNCATE",
)

job = client.load_table_from_dataframe(
    df, f"{PROJECT_ID}.curated.orders", job_config=job_config
)
job.result()   # <-- ALWAYS block on this

print(f"✓ Loaded {job.output_rows} rows, {job.output_bytes:,} bytes")

In [ ]:
# Verify
result = run_to_dataframe(
    f"""
    SELECT order_date, status, COUNT(*) AS n, SUM(total_amount) AS revenue
    FROM `{PROJECT_ID}.curated.orders`
    GROUP BY order_date, status
    ORDER BY order_date, status
    """
)
display(result)

## 5. Nested data — the schema in action

Load a table with a `REPEATED` `RECORD` and query it.

In [ ]:
NESTED_SCHEMA = [
    bigquery.SchemaField("order_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("order_date", "DATE", mode="REQUIRED"),
    bigquery.SchemaField(
        name="line_items", field_type="RECORD", mode="REPEATED",
        fields=[
            bigquery.SchemaField("product_id", "STRING"),
            bigquery.SchemaField("quantity", "INTEGER"),
            bigquery.SchemaField("unit_price", "NUMERIC"),
        ],
    ),
    bigquery.SchemaField("metadata", "JSON"),
]

nested = bigquery.Table(f"{PROJECT_ID}.curated.orders_nested", schema=NESTED_SCHEMA)
nested = client.create_table(nested, exists_ok=True)

rows = [
    {
        "order_id": "o1001", "order_date": "2026-01-15",
        "line_items": [
            {"product_id": "p001", "quantity": 2, "unit_price": 25.0},
            {"product_id": "p002", "quantity": 1, "unit_price": 40.5},
        ],
        "metadata": {"channel": "web", "campaign": "cmp_007"},
    },
    {
        "order_id": "o1002", "order_date": "2026-01-15",
        "line_items": [{"product_id": "p003", "quantity": 1, "unit_price": 99.9}],
        "metadata": {"channel": "mobile"},
    },
]

errors = client.insert_rows_json(nested, rows)
print("Errors:", errors if errors else "none")

In [ ]:
# Query the nested data
result = run_to_dataframe(
    f"""
    SELECT
      order_id,
      item.product_id,
      item.quantity,
      item.unit_price * item.quantity AS line_total,
      JSON_VALUE(metadata, '$.channel') AS channel
    FROM `{PROJECT_ID}.curated.orders_nested`,
    UNNEST(line_items) AS item
    ORDER BY order_id, item.product_id
    """
)
display(result)

## 6. Update the schema

`update_mode="update_fields"` applies only what you set. `"update_full"` replaces
everything and drops what you omitted.

In [ ]:
table = client.get_table(f"{PROJECT_ID}.curated.orders")
print("Before:", [f.name for f in table.schema])

table.schema = [
    *table.schema,
    bigquery.SchemaField("source_channel", "STRING", description="Added 2026-01"),
]
table = client.update_table(table, update_mode="update_fields")
print("After: ", [f.name for f in table.schema])

In [ ]:
# Add a column with SQL instead
run_query(
    f"""
    ALTER TABLE `{PROJECT_ID}.curated.orders`
    ADD COLUMN IF NOT EXISTS discount NUMERIC DEFAULT 0
    """
)

table = client.get_table(f"{PROJECT_ID}.curated.orders")
print("Columns:", [f.name for f in table.schema])

## 7. Views and materialized views

`bigquery.Table` also creates views — set `view_query` or `mview_query`.

In [ ]:
view = bigquery.Table(f"{PROJECT_ID}.curated.v_active_orders")
view.view_query = f"""
    SELECT order_id, customer_id, order_date, total_amount
    FROM `{PROJECT_ID}.curated.orders`
    WHERE status = 'active'
"""
view.description = "Active orders only"
view = client.create_view(view, exists_ok=True)
print(f"✓ View: {view.full_table_id}")

result = run_to_dataframe(f"SELECT COUNT(*) AS n FROM `{view.full_table_id}`")
display(result)

In [ ]:
mv = bigquery.Table(f"{PROJECT_ID}.mart.mv_daily_revenue")
mv.mview_query = f"""
    SELECT order_date, status, COUNT(*) AS order_count, SUM(total_amount) AS revenue
    FROM `{PROJECT_ID}.curated.orders`
    GROUP BY order_date, status
"""
mv.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY, field="order_date"
)
mv.clustering_fields = ["status"]
mv = client.create_table(mv, exists_ok=True)
print(f"✓ MV: {mv.full_table_id}")

result = run_to_dataframe(f"SELECT * FROM `{mv.full_table_id}` LIMIT 5")
display(result)

## 8. Clean up

Deleting a dataset with `delete_contents=True` removes every table inside it. Guard it.

In [ ]:
DELETE_SCRATCH = False   # flip to True to actually delete

if DELETE_SCRATCH:
    try:
        client.delete_dataset(f"{PROJECT_ID}.scratch", delete_contents=True)
        print("✓ Deleted scratch")
    except NotFound:
        print("Already gone")
else:
    print("ℹ️  Set DELETE_SCRATCH = True to actually delete the scratch dataset")

ℹ️  Set DELETE_SCRATCH = True to actually delete the scratch dataset


## 🔑 Takeaways

- `bigquery.Dataset` and `bigquery.Table` mirror the SQL 1:1.
- **`exists_ok=True` is the idempotency key** — always use it in setup code.
- `SchemaField` handles nested and repeated fields via the `fields` argument.
- Partitioning via `TimePartitioning`, clustering via `table.clustering_fields`.
- **Always call `job.result()`** after a load.
- `update_mode="update_fields"` is almost always what you want.

---

## 📚 Next

[03 · Querying and Settings](./03-querying-and-settings.ipynb)